# GRADE RAG Chatbot (Simplified)

Clean RAG pipeline over the GRADE Book — no LangGraph, no agents, no Docker.

**Architecture:**
```
scrape_and_chunk(urls) → build_index() → rag_answer(query)
```

**Setup:** copy `.env.example` to `.env` and fill in:
```
OPENROUTER_API_KEY=...
MODEL_NAME=google/gemini-2.5-flash
```

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from dotenv import load_dotenv
load_dotenv()

from rag_core import load_or_build_index, rag_answer, build_llm

## 1. Configuration

In [2]:
GRADE_URLS = [
    "https://book.gradepro.org/guideline/overview-of-the-grade-approach",
    "https://book.gradepro.org/guideline/the-development-methods-of-grade",
    "https://book.gradepro.org/guideline/requirements-for-claiming-the-use-of-grade",
    "https://book.gradepro.org/guideline/questions-about-interventions-diagnostic-test-prognosis-and-exposures",
    "https://book.gradepro.org/guideline/risk-of-bias-randomized-trials",
    "https://book.gradepro.org/guideline/inconsistency",
    "https://book.gradepro.org/guideline/imprecision",
    "https://book.gradepro.org/guideline/dissemination-bias",
]

GRADE_SYSTEM_PROMPT = (
    "You are an expert assistant specialising in the GRADE methodology for evidence-based "
    "healthcare guidelines. Answer the user's question based ONLY on the retrieved GRADE Book "
    "content below. Do not fabricate or speculate beyond what is in the context. "
    "If the context does not contain enough information, say so explicitly. "
    "Use precise GRADE terminology (certainty of evidence, rating down/up, domains, etc.)."
)

CACHE_DIR = ".faiss_cache/grade"

## 2. Build / Load Index

First run scrapes all 8 GRADE Book pages (~40 s) and saves a local FAISS cache.  
Subsequent runs load from cache instantly.

In [6]:
index = load_or_build_index(GRADE_URLS, cache_path=CACHE_DIR)
llm = build_llm()
print("Ready.")

Building index from scratch...
  Scraping: https://book.gradepro.org/guideline/overview-of-the-grade-approach
    -> 40 chunks
  Scraping: https://book.gradepro.org/guideline/the-development-methods-of-grade
    -> 10 chunks
  Scraping: https://book.gradepro.org/guideline/requirements-for-claiming-the-use-of-grade
    -> 14 chunks
  Scraping: https://book.gradepro.org/guideline/questions-about-interventions-diagnostic-test-prognosis-and-exposures
    -> 15 chunks
  Scraping: https://book.gradepro.org/guideline/risk-of-bias-randomized-trials
    -> 38 chunks
  Scraping: https://book.gradepro.org/guideline/inconsistency
    -> 21 chunks
  Scraping: https://book.gradepro.org/guideline/imprecision
    -> 25 chunks
  Scraping: https://book.gradepro.org/guideline/dissemination-bias
    -> 22 chunks
Total documents: 185


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Index saved to: .faiss_cache/grade
Ready.


## 3. Ask a Question

In [7]:
query = "What are the four levels of certainty of evidence in GRADE?"
answer, contexts = rag_answer(query, index, llm, k=5, system_prompt=GRADE_SYSTEM_PROMPT)
print("Answer:\n", answer)
print("\n--- Retrieved contexts ---")
for i, ctx in enumerate(contexts, 1):
    print(f"[{i}] {ctx[:200]}...")

Answer:
 The provided text does not explicitly list the four levels of certainty of evidence in GRADE. It mentions "GRADE's categories" but does not enumerate them.

--- Retrieved contexts ---
[1]  a particular range."
(5,6)
And for evidence from reviews of qualitative research: “an assessment of the extent to which the review finding is a reasonable representation of the phenomenon of interest...
[2] users to make judgments within these domains. Signalling questions, e.g. about the priority of a problem, are used to answer questions for each criterion.
Requirements for basic GRADE usage
The GRADE ...
[3] ertainty of the evidence therefore begins with a rating of high certainty. The evaluation follows the basic GRADE principles and the five domains for rating down certainty.
(11,12)
In a second step th...
[4] Supported by tools, e.g. GRADEpro
Certainty of evidence
GRADE defines certainty of the evidence as the "certainty that the true value of an estimate falls within a specific range 

## 4. Convenience Functions (for evaluation notebooks)

In [8]:
def ask_grade(question: str) -> str:
    answer, _ = rag_answer(question, index, llm, system_prompt=GRADE_SYSTEM_PROMPT)
    return answer

def get_grade_contexts(question: str, k: int = 5) -> list[str]:
    _, contexts = rag_answer(question, index, llm, k=k, system_prompt=GRADE_SYSTEM_PROMPT)
    return contexts

In [9]:
questions = [
      "What is a potential risk of combining tests with therapeutic interventions?",   # Basic
      "What can a multiple intervention comparison framework help with?",                        # Intermediate
      "What challenge may arise when comparing different diagnostic tests?",  #

  ]
for q in questions:
      ans, ctx = rag_answer(q, index, llm, k=5, system_prompt=GRADE_SYSTEM_PROMPT)
      print(f"Q: {q}\nA: {ans}\n")

Q: What is a potential risk of combining tests with therapeutic interventions?
A: The provided text does not contain information about the potential risks of combining tests with therapeutic interventions. It only states that tests are "almost never used as a single intervention" and that "if a test fails to improve outcomes there is no reason to use it, whatever its accuracy."

Q: What can a multiple intervention comparison framework help with?
A: A multiple intervention comparison framework and a statistical summary via network meta-analysis can help address the complexity that arises from the existence of multiple alternatives to an intervention.

Q: What challenge may arise when comparing different diagnostic tests?
A: When comparing different diagnostic tests, a challenge that may arise is the lack of direct evidence about the effects on relevant outcomes of different test or test strategies. This is because tests are almost never used as a single intervention, but rather as part 

## 5. Optional Gradio UI

In [ ]:
import gradio as gr

def gradio_fn(question, history):
    answer, _ = rag_answer(question, index, llm, system_prompt=GRADE_SYSTEM_PROMPT)
    return answer

demo = gr.ChatInterface(
    fn=gradio_fn,
    title="GRADE Book Assistant",
    description="Ask questions about the GRADE methodology. Answers are grounded in the GRADE Book.",
)
# Uncomment to launch:
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/martynabaran/Downloads/agentic-rag-main/.venv/lib/python3.11/site-packages/gradio/queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/martynabaran/Downloads/agentic-rag-main/.venv/lib/python3.11/site-packages/gradio/route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/martynabaran/Downloads/agentic-rag-main/.venv/lib/python3.11/site-packages/gradio/blocks.py", line 2158, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/martynabaran/Downloads/agentic-rag-main/.venv/lib/python3.11/site-packages/gradio/blocks.py", line 1632, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/martynabaran/Downloads/agenti